In [1]:
# importing  libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

In [2]:
# amazon laptop search page
URL = "https://www.amazon.in/s?k=laptops"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
    "Accept-Language": "en-US, en;q=0.5"
}

# sending request
response = requests.get(URL, headers=HEADERS)

In [3]:
# checking response
print(response.status_code)

# parsing html
soup = BeautifulSoup(response.content, "html.parser")

print(soup.title.text)

200
Amazon.in : laptops


In [4]:
if response.status_code != 200:
    print("Failed to fetch webpage")
    exit()

In [5]:
# finding all laptops cards on the page
laptops = soup.find_all("div", {"data-component-type": "s-search-result"})

# checking how many laptops were found
print("laptops found:", len(laptops))

laptops found: 22


In [6]:
data = []

for laptop in laptops:
    product_info = {}

    # Image URL
    image = laptop.find("img")
    product_info["Image"] = image.get("src") if image else None

    # Title
    title = laptop.find("h2")
    product_info["Title"] = title.get_text(strip=True) if title else None

    # Rating
    rating = laptop.find("span", class_="a-icon-alt")
    product_info["Rating"] = rating.get_text(strip=True) if rating else None

    # Price
    price = laptop.find("span", class_="a-price-whole")
    product_info["Price"] = price.get_text(strip=True) if price else None

    # Ad / Organic
    sponsored = laptop.find(string=lambda text: text and "Sponsored" in text)
    product_info["Ad / Organic"] = "Ad" if sponsored else "Organic"

    data.append(product_info)



In [7]:
# Converting the  data into  DataFrame
df = pd.DataFrame(data)

#  first 4 rows of the DataFrame
df.head(4)

,Image,Title,Rating,Price,Ad / Organic
0,https://m.media-amazon.com/images/I/71MAe4Du8u...,"ASUS Vivobook S16,Intel Core Ultra 5 225H,AI P...",4.1 out of 5 stars,"79,990",Ad
1,https://m.media-amazon.com/images/I/71ic0yMZhT...,HP OmniBook AMD Ryzen 5 New AI 330 Processor (...,4.9 out of 5 stars,"69,990",Ad
2,https://m.media-amazon.com/images/I/71ouu-iX3p...,"Acer Smartchoice Aspire One, AMD Ryzen 3-7320U...",5.0 out of 5 stars,"39,990",Organic
3,https://m.media-amazon.com/images/I/717WZ7Wriw...,"Dell 15 (Previously Inspiron) Laptop, 14th Gen...",4.1 out of 5 stars,"46,490",Organic


In [8]:
#Converting dataframe into csv file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
df.to_csv(
    f"amazon_laptops_{timestamp}.csv",
    index=False,
    encoding="utf-8-sig"
)